In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06_feature_correlation_pca_reduction
# MAGIC 
# MAGIC **Objetivo**: Reducción dimensional mediante:
# MAGIC 1. Eliminación de features altamente correlacionadas (|corr| > 0.85)
# MAGIC 2. PCA automático para alcanzar varianza acumulada ≥ 85%
# MAGIC 
# MAGIC **Periodo**: 2016-09-04 hasta 2018-09-30 23:59:59
# MAGIC 
# MAGIC **Salidas**:
# MAGIC - Dataset reducido: `/Volumes/olist/gold/customer_features_rfm_20180930_reduced`
# MAGIC - Métricas: features eliminadas y varianza PCA

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Paths
GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"

print("=" * 80)
print("🚀 INICIO: REDUCCIÓN POR CORRELACIÓN + PCA AUTOMÁTICO")
print("=" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Verificación del entorno

# COMMAND ----------

print("🔍 Verificando entorno de Databricks...")
print()

# Mostrar catálogo actual
try:
    current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    current_schema = spark.sql("SELECT current_schema()").collect()[0][0]
    print(f"📂 Catálogo actual: {current_catalog}")
    print(f"📂 Esquema actual: {current_schema}")
except Exception as e:
    print(f"⚠️  No se pudo obtener catálogo/esquema actual: {e}")

print()

# Listar catálogos disponibles
try:
    catalogs = spark.sql("SHOW CATALOGS").toPandas()
    print(f"📚 Catálogos disponibles ({len(catalogs)}):")
    for cat in catalogs['catalog'].head(10):
        print(f"   • {cat}")
except Exception as e:
    print(f"⚠️  Error listando catálogos: {e}")

print()
print("-" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Preparación de estructura de catálogo

# COMMAND ----------

print("🔧 Verificando/creando estructura de Unity Catalog...")
print()

# Crear catálogo si no existe
try:
    spark.sql("CREATE CATALOG IF NOT EXISTS olist")
    print("✅ Catálogo 'olist' verificado/creado")
except Exception as e:
    print(f"⚠️  Catálogo: {e}")

# Crear esquema olist_gold si no existe
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS olist.olist_gold")
    print("✅ Esquema 'olist.olist_gold' verificado/creado")
except Exception as e:
    print(f"⚠️  Esquema: {e}")

# Crear volumes si no existen
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.gold")
    print("✅ Volume 'gold' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume gold: {e}")

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.metrics")
    print("✅ Volume 'metrics' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume metrics: {e}")

print()
print("✅ Estructura de Unity Catalog lista")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Carga de datos

# COMMAND ----------

print("📥 Cargando features desde GOLD...")
source_path = f"{GOLD_PATH}customer_features_rfm_20180930/"
print(f"   Fuente: {source_path}")
print()

# Verificar que exista la tabla
try:
    files = dbutils.fs.ls(source_path)
    print(f"✅ Directorio encontrado con {len(files)} archivos")
except Exception as e:
    print(f"❌ ERROR: El directorio fuente no existe")
    print(f"   Ruta: {source_path}")
    print(f"   Error: {e}")
    print()
    print("⚠️  ACCIÓN REQUERIDA:")
    print("   Debes ejecutar primero el notebook de generación de features RFM")
    print("   (05_customer_features_engineering)")
    raise

# Cargar datos
try:
    df = spark.read.format("delta").load(source_path).toPandas()
    print(f"✅ Dataset cargado exitosamente")
    print(f"   - Registros: {len(df):,}")
    print(f"   - Columnas: {len(df.columns)}")
    print()
except Exception as e:
    print(f"❌ Error cargando datos: {e}")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Preparación de datos

# COMMAND ----------

print("🎯 Separando features y target...")
print()

# Columnas a excluir de las features
exclude_cols = ["customer_id", "is_premium"]

# Si existe cluster_ordered, también excluirlo
if "cluster_ordered" in df.columns:
    exclude_cols.append("cluster_ordered")

# Identificar features
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Separar X y y
X_original = df[feature_cols].copy()
y = df["is_premium"].copy()
customer_id = df["customer_id"].copy()

print(f"✅ Separación completada:")
print(f"   - Features originales: {len(feature_cols)}")
print(f"   - Target: is_premium ({y.sum():,} premium / {len(y):,} total)")
print(f"   - Tasa premium: {y.mean()*100:.2f}%")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Manejo de valores faltantes y categóricos

# COMMAND ----------

print("🔧 Procesando features...")
print()

# Rellenar NaN con 0
X = X_original.fillna(0)
nan_count = X_original.isna().sum().sum()
if nan_count > 0:
    print(f"   ℹ️  Valores NaN rellenados con 0: {nan_count:,}")

# Convertir categóricas a dummies si existen
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_cols) > 0:
    print(f"   ℹ️  Variables categóricas encontradas: {len(categorical_cols)}")
    print(f"      {categorical_cols}")
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f"   ✅ Dummies creados. Nuevas features: {len(X.columns)}")
else:
    print(f"   ✅ No hay variables categóricas")

print(f"\n✅ Features finales para análisis: {len(X.columns)}")
print()

# Actualizar lista de features
feature_cols = X.columns.tolist()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Análisis de correlación

# COMMAND ----------

print("📊 PASO 1: ANÁLISIS Y ELIMINACIÓN POR CORRELACIÓN")
print("=" * 80)
print()

print("Calculando matriz de correlación...")
corr_matrix = X.corr()
print(f"✅ Matriz calculada: {corr_matrix.shape[0]} x {corr_matrix.shape[1]}")
print()

# Umbral de correlación
threshold = 0.85

print(f"Identificando pares con |correlación| > {threshold}...")
print()

# Identificar pares altamente correlacionados
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > threshold:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            high_corr_pairs.append((col1, col2, corr_val))

print(f"✅ Pares encontrados: {len(high_corr_pairs)}")
print()

if len(high_corr_pairs) > 0:
    print("Primeros 10 pares con mayor correlación:")
    sorted_pairs = sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)
    for col1, col2, corr in sorted_pairs[:10]:
        print(f"   • {col1} <-> {col2}: {corr:.4f}")
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Eliminación de features correlacionadas

# COMMAND ----------

print("🗑️  Eliminando features redundantes...")
print("   Criterio: mantener variable alfabéticamente menor")
print()

# Determinar qué variables eliminar
to_drop = set()
for col1, col2, _ in high_corr_pairs:
    # Mantener la primera alfabéticamente
    if col1 < col2:
        to_drop.add(col2)
    else:
        to_drop.add(col1)

to_drop = sorted(list(to_drop))
features_retained = [c for c in feature_cols if c not in to_drop]

print(f"✅ Resumen:")
print(f"   - Features originales: {len(feature_cols)}")
print(f"   - Features eliminadas: {len(to_drop)}")
print(f"   - Features retenidas: {len(features_retained)}")
print(f"   - Reducción: {len(to_drop)/len(feature_cols)*100:.2f}%")
print()

# Aplicar reducción
X_reduced = X[features_retained].copy()

print(f"✅ Dataset reducido: {X_reduced.shape[0]:,} x {X_reduced.shape[1]}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Guardar features eliminadas

# COMMAND ----------

print("💾 Guardando features eliminadas...")
print()

if len(to_drop) > 0:
    dropped_df = pd.DataFrame({
        'feature': to_drop,
        'reason': 'correlation > 0.85'
    })
    
    # Guardar usando método temporal para CSV
    temp_path = f"{METRICS_PATH}features_temp/"
    spark.createDataFrame(dropped_df).write \
        .format("csv").mode("overwrite").option("header", "true") \
        .save(temp_path)
    
    # Mover el archivo CSV
    csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
    if csv_files:
        dbutils.fs.cp(csv_files[0].path, f"{METRICS_PATH}features_correlacion_eliminadas.csv")
    
    dbutils.fs.rm(temp_path, True)
    print(f"✅ Guardado: features_correlacion_eliminadas.csv")
else:
    print("ℹ️  No hay features eliminadas por correlación")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Estandarización

# COMMAND ----------

print("📏 PASO 2: ESTANDARIZACIÓN")
print("=" * 80)
print()

print("Aplicando StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced)

print(f"✅ Features estandarizadas")
print(f"   - Shape: {X_scaled.shape}")
print(f"   - Media ~0, Std ~1")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. PCA Automático (Varianza ≥ 85%)

# COMMAND ----------

print("🔬 PASO 3: PCA AUTOMÁTICO")
print("=" * 80)
print()

print("Objetivo: Varianza acumulada ≥ 85%")
print()

# Ajustar PCA con todos los componentes posibles
print("Calculando varianza explicada por todos los componentes...")
pca_full = PCA()
pca_full.fit(X_scaled)

variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

print(f"✅ Componentes totales disponibles: {len(variance_ratio)}")
print()

# Encontrar número de componentes para ≥ 85%
target_variance = 0.85
n_components = np.argmax(cumulative_variance >= target_variance) + 1

print(f"📊 Resultado:")
print(f"   - Componentes necesarios: {n_components}")
print(f"   - Varianza acumulada: {cumulative_variance[n_components-1]*100:.2f}%")
print(f"   - Reducción dimensional: {len(feature_cols)} → {n_components}")
print(f"   - Factor de reducción: {n_components/len(feature_cols)*100:.2f}%")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Aplicar PCA con componentes seleccionados

# COMMAND ----------

print("Aplicando PCA con componentes seleccionados...")
print()

# PCA final
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

print(f"✅ Transformación PCA completada")
print(f"   - Shape resultante: {X_pca.shape}")
print()

# Mostrar varianza por componente
print("Varianza explicada por componente:")
for i in range(min(10, n_components)):
    print(f"   PCA_{i+1}: {pca.explained_variance_ratio_[i]*100:.2f}% "
          f"(acum: {cumulative_variance[i]*100:.2f}%)")
if n_components > 10:
    print(f"   ... ({n_components - 10} componentes adicionales)")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Crear dataset final

# COMMAND ----------

print("📦 Creando dataset final reducido...")
print()

# Crear columnas PCA
pca_cols = [f"pca_{i+1}" for i in range(n_components)]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)

# Agregar customer_id y target
df_pca['customer_id'] = customer_id.values
df_pca['is_premium'] = y.values

print(f"✅ Dataset final creado:")
print(f"   - Registros: {len(df_pca):,}")
print(f"   - Columnas PCA: {n_components}")
print(f"   - Columnas totales: {len(df_pca.columns)} (PCA + customer_id + is_premium)")
print()

# Verificar balance del target
print(f"Balance del target:")
print(f"   - Premium: {df_pca['is_premium'].sum():,} ({df_pca['is_premium'].mean()*100:.2f}%)")
print(f"   - No premium: {(~df_pca['is_premium']).sum():,} ({(~df_pca['is_premium']).mean()*100:.2f}%)")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Guardar dataset reducido

# COMMAND ----------

print("💾 Guardando dataset reducido en GOLD...")
print()

output_path = f"{GOLD_PATH}customer_features_rfm_20180930_reduced/"

spark.createDataFrame(df_pca) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(output_path)

print(f"✅ Dataset guardado exitosamente")
print(f"   Ubicación: {output_path}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. Guardar métricas de PCA

# COMMAND ----------

print("💾 Guardando métricas de varianza PCA...")
print()

# Crear DataFrame con varianza
pca_variance_df = pd.DataFrame({
    'component': [f'pca_{i+1}' for i in range(n_components)],
    'variance_explained': pca.explained_variance_ratio_,
    'cumulative_variance': cumulative_variance[:n_components],
    'variance_pct': pca.explained_variance_ratio_ * 100,
    'cumulative_pct': cumulative_variance[:n_components] * 100
})

# Guardar usando método temporal
temp_path = f"{METRICS_PATH}pca_temp/"
spark.createDataFrame(pca_variance_df).write \
    .format("csv").mode("overwrite").option("header", "true") \
    .save(temp_path)

# Mover el archivo CSV
csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
if csv_files:
    dbutils.fs.cp(csv_files[0].path, f"{METRICS_PATH}pca_varianza_explicada.csv")

dbutils.fs.rm(temp_path, True)

print(f"✅ Guardado: pca_varianza_explicada.csv")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 14. Visualizaciones

# COMMAND ----------

print("📈 Generando visualizaciones...")
print()

# Crear figura con 3 subplots
fig = plt.figure(figsize=(18, 5))

# 1. Scree Plot
ax1 = plt.subplot(131)
components_range = range(1, n_components + 1)
ax1.bar(components_range, pca.explained_variance_ratio_, 
        alpha=0.7, color='steelblue', label='Varianza individual')
ax1.plot(components_range, cumulative_variance[:n_components], 
         'ro-', linewidth=2, markersize=8, label='Varianza acumulada')
ax1.axhline(y=0.85, color='red', linestyle='--', alpha=0.7, 
            linewidth=2, label='Target: 85%')
ax1.set_xlabel('Componente Principal', fontsize=11)
ax1.set_ylabel('Varianza Explicada', fontsize=11)
ax1.set_title('Scree Plot - PCA', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_xticks(components_range)

# 2. Heatmap de correlación (muestra)
ax2 = plt.subplot(132)
if len(features_retained) > 25:
    # Muestra aleatoria
    sample_size = 25
    np.random.seed(42)
    sample_features = np.random.choice(features_retained, sample_size, replace=False)
    corr_sample = X_reduced[sample_features].corr()
    title = f'Matriz de Correlación\n(Muestra: {sample_size}/{len(features_retained)} features)'
else:
    corr_sample = X_reduced.corr()
    title = f'Matriz de Correlación\n({len(features_retained)} features retenidas)'

sns.heatmap(corr_sample, cmap='RdBu_r', center=0, ax=ax2,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, xticklabels=False, yticklabels=False)
ax2.set_title(title, fontsize=13, fontweight='bold')

# 3. Reducción dimensional
ax3 = plt.subplot(133)
stages = ['Original', 'Post-Correlación', 'Post-PCA']
dimensions = [len(feature_cols), len(features_retained), n_components]
colors = ['#e74c3c', '#f39c12', '#27ae60']

bars = ax3.bar(stages, dimensions, color=colors, alpha=0.7, edgecolor='black')
ax3.set_ylabel('Número de Dimensiones', fontsize=11)
ax3.set_title('Reducción Dimensional', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# Agregar valores en las barras
for bar, dim in zip(bars, dimensions):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(dim)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 15. Resumen final

# COMMAND ----------

print()
print("=" * 80)
print("✅ REDUCCIÓN DIMENSIONAL COMPLETADA EXITOSAMENTE")
print("=" * 80)
print()

print("📊 RESUMEN DE REDUCCIÓN:")
print("-" * 80)
print(f"Features originales:              {len(feature_cols):>6}")
print(f"Features eliminadas (correlación): {len(to_drop):>6}")
print(f"Features retenidas:                {len(features_retained):>6}")
print(f"Componentes PCA seleccionados:     {n_components:>6}")
print(f"Varianza acumulada final:          {cumulative_variance[n_components-1]*100:>6.2f}%")
print()

print("📉 REDUCCIÓN TOTAL:")
print("-" * 80)
reduction_pct = (1 - n_components/len(feature_cols)) * 100
print(f"Dimensiones: {len(feature_cols)} → {n_components}")
print(f"Factor de reducción: {reduction_pct:.2f}%")
print()

print("💾 OUTPUTS GENERADOS:")
print("-" * 80)
print(f"✓ Dataset reducido (Delta):")
print(f"  {output_path}")
print(f"  • Registros: {len(df_pca):,}")
print(f"  • Columnas: {len(df_pca.columns)}")
print()
print(f"✓ Features eliminadas (CSV):")
print(f"  {METRICS_PATH}features_correlacion_eliminadas.csv")
print(f"  • Features: {len(to_drop)}")
print()
print(f"✓ Varianza PCA (CSV):")
print(f"  {METRICS_PATH}pca_varianza_explicada.csv")
print(f"  • Componentes: {n_components}")
print()

print("🎯 SIGUIENTE PASO:")
print("-" * 80)
print("Dataset listo para split train/validation/test (Prompt 7)")
print()
print("=" * 80)